# 08. Seq2Seq — 문장을 문장으로 번역하기

노트북 06에서 RNN으로 순서 데이터를 다뤘다. 이번엔 한 걸음 더 나아간다.
**입력도 시퀀스, 출력도 시퀀스** — 예를 들어 영어 문장을 한국어 문장으로 **번역**한다.

```
'i love you'  →  [RNN 인코더-디코더]  →  '나는 당신을 사랑합니다'
  영어 시퀀스                              한국어 시퀀스
```

이런 구조를 **Seq2Seq(Sequence-to-Sequence)** 라 한다. 번역·요약·챗봇의 기본 뼈대다.

이 노트북에서 배우는 것:
1. **텍스트를 숫자로** — 단어 사전을 손으로 만들기 → `Tokenizer`로 자동화
2. **`pad_sequences`** — 길이가 다른 문장을 맞추기
3. **Seq2Seq 번역** — 영↔한 8문장으로 번역 모델
4. **Bidirectional, LSTM** — RNN을 강화하는 방법 (노트북 06의 SimpleRNN 확장)

> 노트북 06에서 `SimpleRNN`, `return_sequences`, 원-핫을 다뤘다. 여기서는 그걸 전제로 한다.


## 1부. 텍스트를 숫자로 — 손으로 만들어보기

모델은 글자를 못 받는다. 단어를 숫자로 바꿔야 한다. 먼저 **원리를 이해하기 위해 손으로** 만든다.

번역할 데이터: 같은 뜻의 영어 4문장 + 한국어 4문장.


In [1]:
docs = ['i love you',
        'i hate you',
        'i like you',
        'i curse you',
        '나는 당신을 사랑합니다',
        '나는 당신을 미워합니다',
        '나는 당신을 좋아합니다',
        '나는 당신을 저주합니다']

# 단어 사전을 손으로 만들기: 모든 문장의 단어를 모아 번호 부여
wd2idx = {}
for sent in docs:
    for wd in sent.split():
        if wd not in wd2idx:
            wd2idx[wd] = len(wd2idx)
print(wd2idx)


{'i': 0, 'love': 1, 'you': 2, 'hate': 3, 'like': 4, 'curse': 5, '나는': 6, '당신을': 7, '사랑합니다': 8, '미워합니다': 9, '좋아합니다': 10, '저주합니다': 11}


### 1-1. 손으로 만든 사전의 한계

위 방식은 되긴 하지만 불편하다:
- 번호가 0부터 시작 → 나중에 "빈 자리(padding)"를 0으로 채울 때 실제 단어와 충돌
- 대소문자, 문장부호 처리를 직접 해야 함
- 문장 길이가 다르면 맞춰줄 방법이 없음

Keras의 **`Tokenizer`** 가 이 모든 걸 자동으로 해준다. 다음 셀부터 그걸 쓴다.


In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer

tknzr = Tokenizer()
tknzr.fit_on_texts(docs)
print("단어→번호:", tknzr.word_index)
print("번호→단어:", tknzr.index_word)


단어→번호: {'i': 1, 'you': 2, '나는': 3, '당신을': 4, 'love': 5, 'hate': 6, 'like': 7, 'curse': 8, '사랑합니다': 9, '미워합니다': 10, '좋아합니다': 11, '저주합니다': 12}
번호→단어: {1: 'i', 2: 'you', 3: '나는', 4: '당신을', 5: 'love', 6: 'hate', 7: 'like', 8: 'curse', 9: '사랑합니다', 10: '미워합니다', 11: '좋아합니다', 12: '저주합니다'}


### 1-2. 문장을 숫자 시퀀스로 + 길이 맞추기 (`pad_sequences`)

`texts_to_sequences` : 문장을 단어 번호의 리스트로 바꾼다.
`pad_sequences(..., 4)` : 모든 시퀀스를 **길이 4로 통일**한다. 짧으면 **앞에 0을 채운다**(pre-padding).

```
'i love you'  →  [1, 5, 2]  →  [0, 1, 5, 2]   ← 앞에 0 하나 추가
```

왜 길이를 맞추나? 신경망은 **고정된 크기의 입력**만 받기 때문이다. 문장마다 길이가 다르면 안 된다.


In [3]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

encd = tknzr.texts_to_sequences(docs)
print("숫자 시퀀스:", encd)

pad_encd = pad_sequences(encd, 4)     # 길이 4로 통일 (앞에 0 채움)
print("패딩 후:\n", pad_encd)


숫자 시퀀스: [[1, 5, 2], [1, 6, 2], [1, 7, 2], [1, 8, 2], [3, 4, 9], [3, 4, 10], [3, 4, 11], [3, 4, 12]]
패딩 후:
 [[ 0  1  5  2]
 [ 0  1  6  2]
 [ 0  1  7  2]
 [ 0  1  8  2]
 [ 0  3  4  9]
 [ 0  3  4 10]
 [ 0  3  4 11]
 [ 0  3  4 12]]


### 1-3. 원-핫 인코딩과 입력/정답 분리

각 단어 번호를 원-핫으로 바꾼다. 단어가 12개 + 패딩(0) = **13차원**.

그리고 데이터를 나눈다:
- **x_data** : 앞 4개(영어 문장) — 번역의 **입력**
- **y_data** : 뒤 4개(한국어 문장) — 번역의 **정답**

`(4, 4, 13)` = 4문장 × 4단어 × 13차원.


In [4]:
from tensorflow.keras.utils import to_categorical

data_oh = to_categorical(pad_encd)    # (8, 4, 13)
x_data = data_oh[:4]                   # 영어 4문장
y_data = data_oh[4:]                   # 한국어 4문장
print("x (영어):", x_data.shape, "| y (한국어):", y_data.shape)


x (영어): (4, 4, 13) | y (한국어): (4, 4, 13)


## 2부. Seq2Seq 번역 모델

이제 영어를 넣으면 한국어가 나오는 모델을 만든다.

```
Input(4, 13)                              # 4단어 × 13차원
  → SimpleRNN(16, return_sequences=True)  # 각 단어 위치의 출력을 유지
  → SimpleRNN(32, return_sequences=True)
  → Dense(13, softmax)                    # 각 위치마다 한국어 단어 예측
```

- `return_sequences=True` (두 RNN 다) : **각 위치마다 출력**이 필요하다.
  번역은 "입력 4단어 → 출력 4단어"이므로, 위치별로 한국어 단어를 내야 한다. (노트북 06 문자예측과 같은 원리)
- `Model(...)` (함수형 API) : `Sequential` 대신 층을 함수처럼 연결한다.
  입력·출력을 명시적으로 지정 — 나중에 인코더/디코더를 나눌 때 유용하다.

**학습: `fit(x_data, y_data)`** — 영어(x)를 넣으면 한국어(y)가 나오게.


In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, SimpleRNN, Dense

input_layer = Input(shape=(4, 13))
rnn_1 = SimpleRNN(16, return_sequences=True)(input_layer)
rnn_2 = SimpleRNN(32, return_sequences=True)(rnn_1)
output_layer = Dense(13, activation='softmax')(rnn_2)

model = Model(input_layer, output_layer)
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_data, y_data, epochs=100, verbose=0)   # 로그 생략
print("학습 완료")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4, 13)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 4, 16)          │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 4, 32)          │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4, 13)          │           429 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,477 (9.68 KB)

 Trainable params: 2,477 (9.68 KB)

 Non-trainable params: 0 (0.00 B)

학습 완료


### 2-1. 번역 함수 — 영어를 넣으면 한국어가

학습한 모델로 실제 번역을 해본다. 과정은 학습 때와 똑같다:
1. 영어 문장 → `texts_to_sequences` → `pad_sequences` → 원-핫
2. 모델 예측 → 각 위치의 최댓값(argmax) = 한국어 단어 번호
3. 번호를 단어로 되돌려 이어붙임

패딩(0) 위치는 건너뛴다.


In [6]:
import numpy as np

def my_translate(txt):
    encd = tknzr.texts_to_sequences([txt])
    pad = pad_sequences(encd, 4)
    x = to_categorical(pad, 13)
    pred = model.predict(x, verbose=0)
    ids = np.argmax(pred, axis=2)[0]           # 각 위치의 예측 단어 번호
    words = [tknzr.index_word[int(i)] for i in ids if i != 0]   # 0(패딩)은 제외
    return ' '.join(words)

for txt in ['i love you', 'i hate you', 'i like you', 'i curse you']:
    print(f"{txt:15s} => {my_translate(txt)}")


i love you      => 나는 당신을 사랑합니다
i hate you      => 나는 당신을 미워합니다
i like you      => 나는 당신을 좋아합니다
i curse you     => 나는 당신을 저주합니다


## 3부. RNN을 강화하기 — Bidirectional과 LSTM

앞의 번역 모델은 `SimpleRNN`을 썼다. 이걸 더 강하게 만드는 두 가지 방법을 본다.
(노트북 06 마지막에서 "LSTM/GRU로 바꾸면 된다"고 예고했던 부분이다.)

### 3-1. Bidirectional — 양방향으로 읽기

`SimpleRNN`은 문장을 **앞→뒤 한 방향**으로만 읽는다. 그런데 언어는 **뒤 단어가 앞 단어의 의미를 바꾸기도** 한다.

> "그는 은행(bank)에 갔다" — 뒤에 "돈을 찾으러"가 오는지 "낚시하러"가 오는지에 따라 은행의 뜻이 다르다.

**`Bidirectional`** 은 문장을 **앞→뒤, 뒤→앞 양방향으로 동시에 읽어** 두 기억을 합친다.
그래서 각 단어를 판단할 때 **앞뒤 문맥을 모두** 본다. 대신 파라미터는 2배가 된다.

```python
Bidirectional(SimpleRNN(16, return_sequences=True))
```


In [7]:
from tensorflow.keras.layers import Bidirectional

input_layer = Input(shape=(4, 13))
rnn_1 = Bidirectional(SimpleRNN(16, return_sequences=True))(input_layer)
rnn_2 = Bidirectional(SimpleRNN(32, return_sequences=True))(rnn_1)
output_layer = Dense(13, activation='softmax')(rnn_2)

model_bi = Model(input_layer, output_layer)
model_bi.summary()

model_bi.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_bi.fit(x_data, y_data, epochs=200, verbose=0)
print("Bidirectional 학습 완료")


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 4, 13)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 4, 32)          │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 4, 64)          │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4, 13)          │           845 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,965 (23.30 KB)

 Trainable params: 5,965 (23.30 KB)

 Non-trainable params: 0 (0.00 B)

Bidirectional 학습 완료


### 3-2. LSTM — 긴 기억을 위한 게이트

`SimpleRNN`은 긴 문장에서 **앞부분 기억을 잃는** 약점이 있다 (노트북 06에서 언급한 기울기 소실).

**`LSTM`(Long Short-Term Memory)** 은 "무엇을 기억하고 무엇을 잊을지"를 조절하는
**게이트(gate)** 를 가진 개선된 RNN이다. 긴 시퀀스에서 훨씬 안정적이다.

Keras에서는 `SimpleRNN`을 `LSTM`으로 **바꾸기만** 하면 된다. 여기선 Bidirectional과 함께 쓴다.

```python
Bidirectional(LSTM(16, return_sequences=True))
```

> LSTM은 게이트가 여러 개라 파라미터가 SimpleRNN보다 약 4배 많다. 그만큼 표현력이 크다.


In [8]:
from tensorflow.keras.layers import LSTM

input_layer = Input(shape=(4, 13))
rnn_1 = Bidirectional(LSTM(16, return_sequences=True))(input_layer)
rnn_2 = Bidirectional(LSTM(32, return_sequences=True))(rnn_1)
output_layer = Dense(13, activation='softmax')(rnn_2)

model_lstm = Model(input_layer, output_layer)
model_lstm.summary()

model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.fit(x_data, y_data, epochs=200, verbose=0)
print("Bidirectional LSTM 학습 완료")


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 4, 13)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 4, 32)          │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 4, 64)          │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4, 13)          │           845 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,325 (83.30 KB)

 Trainable params: 21,325 (83.30 KB)

 Non-trainable params: 0 (0.00 B)

Bidirectional LSTM 학습 완료


### 3-3. 세 모델 비교 ★

같은 번역 문제를 세 가지 방식으로 풀었다. 실행 결과:

| 모델 | 구조 | 파라미터 | 번역 결과 |
|---|---|---|---|
| 1부 | SimpleRNN ×2 | **2,477** | 나는 당신을 사랑합니다 ✓ |
| Bidirectional | Bi-SimpleRNN ×2 | **5,965** (2.4배) | 나는 당신을 사랑합니다 ✓ |
| LSTM | Bi-LSTM ×2 | **21,325** (8.6배) | 나는 당신을 사랑합니다 ✓ |

데이터가 4문장뿐이라 **셋 다 결국 외운다**(번역 성공). 그래서 여기서 셋의 차이는
**성능이 아니라 표현력과 비용**이다:

- **Bidirectional**은 양방향이라 RNN 부분이 2배 → 파라미터 2.4배. 앞뒤 문맥을 함께 본다.
- **LSTM**은 게이트가 여러 개라 SimpleRNN보다 4배 무겁다 → 여기에 양방향까지 더해 8.6배.
- 짧고 단순한 문제(이 4문장)에는 SimpleRNN으로 충분하다.
- **길고 복잡한 실제 문장**(번역·챗봇·요약)일수록 Bidirectional + LSTM의 표현력이 필요하다.

> **교훈: 도구는 문제에 맞게 고른다.** 4문장에 Bi-LSTM은 과하고,
> 실제 번역에 SimpleRNN은 부족하다. (노트북 06의 "1층 vs 2층"과 같은 맥락 — 크다고 항상 좋은 게 아니다.)

---

## 정리 — Seq2Seq와 텍스트 다루기

| 개념 | 핵심 |
|---|---|
| **Tokenizer** | 단어를 번호로 자동 변환. **번호는 1부터**(0은 padding 전용) |
| **pad_sequences** | 길이가 다른 문장을 고정 길이로. 앞에 0을 채운다(pre-padding) |
| **Seq2Seq** | 입력 시퀀스 → 출력 시퀀스. `return_sequences=True` + 함수형 `Model` |
| **Bidirectional** | 앞→뒤, 뒤→앞 양방향으로 읽어 앞뒤 문맥을 모두 본다 (파라미터 2배) |
| **LSTM** | 게이트로 긴 기억을 유지. SimpleRNN의 기울기 소실을 해결 (파라미터 4배) |

**다음 노트북(09)** 에서는 모델에 넣기 *전* 단계 —
**텍스트 전처리**를 본다. 표제어 추출, 어간 추출, 불용어 제거(영어),
그리고 한국어 형태소 분석(KoNLPy)까지. 실전 NLP는 이 전처리가 절반이다.

In [9]:
import numpy as np

def translate_with(m, txt):
    encd = tknzr.texts_to_sequences([txt])
    pad = pad_sequences(encd, 4)
    x = to_categorical(pad, 13)
    ids = np.argmax(m.predict(x, verbose=0), axis=2)[0]
    return ' '.join(tknzr.index_word[int(i)] for i in ids if i != 0)

print("1부 파라미터:", model.count_params())
print("Bi  파라미터:", model_bi.count_params())
print("LSTM파라미터:", model_lstm.count_params())
print()
txt = 'i love you'
print(f"입력: {txt}")
print("  SimpleRNN   =>", translate_with(model, txt))
print("  Bidirectional =>", translate_with(model_bi, txt))
print("  Bi-LSTM     =>", translate_with(model_lstm, txt))


1부 파라미터: 2477
Bi  파라미터: 5965
LSTM파라미터: 21325

입력: i love you
  SimpleRNN   => 나는 당신을 사랑합니다
  Bidirectional => 나는 당신을 사랑합니다
  Bi-LSTM     => 나는 당신을 사랑합니다
